# 07 — Random Survival Forest (improved): years until a bridge leaves Good

**Primary objective:** median **remaining years** until the first Good to Fair/Poor drop, conditional on the bridge having stayed Good for `entry_duration` years.

**What changed from the earlier RSF notebook** (chosen in Notebook 13 on the 2016 historical validation, not on the 2021 test set): added the deck, superstructure, substructure, culvert and overall condition ratings from each bridge's first Good year. Notebook 13 validation C-index: 0.6275 (earlier setup) to 0.6391 (this setup, same forest settings). More trees and deeper trees added only about +0.004 and state added about +0.002, both within noise, so they are left out.

**Evaluation:** ranking C-index on the **entire** aligned test set using the forest's risk score. It measures ranking, not error in years, and does not adjust for delayed entry. A median is returned only where the learned survival curve reaches 50% within the observed training time; other bridges are left blank, not guessed.

In [1]:
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore", category=FutureWarning)

# Project folders (same logic as the other notebooks).
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if PROJECT.name == "bridge_time_primary_notebooks":
    PROJECT = PROJECT.parent
FINAL = PROJECT / "data" / "final"
PROCESSED = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

RARE_MIN = 200    # categories rarer than this are grouped as "other"

import gc
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

# Forest settings (same as the earlier RSF notebook).
# Notebook 13 also tested 100 trees, depth 16-20, leaf 25: about +0.004, within noise.
N_ESTIMATORS = 50
MAX_DEPTH = 12
MIN_SAMPLES_LEAF = 50
MIN_SAMPLES_SPLIT = 100
N_JOBS = 2
RANDOM_STATE = 42
BATCH_SIZE = 100
print("Project directory:", PROJECT)

Project directory: /Users/annie/Downloads/bridge-deterioration


## 1. Load data and add condition ratings

In [2]:
EXTRA_SRC = ["bridge_id", "STATE_CODE_001", "DECK_COND_058", "SUPERSTRUCTURE_COND_059",
             "SUBSTRUCTURE_COND_060", "CULVERT_COND_062", "condition_rating"]
RATING_COLS = ["deck_rating", "super_rating", "sub_rating", "culvert_rating", "min_rating"]

def add_baseline_extras(df):
    # Attach state and ratings recorded in each bridge's first Good year (its start_year).
    parts = []
    for yr, grp in df.groupby("start_year"):
        snap = pd.read_parquet(PROCESSED / f"nbi_{int(yr)}_clean.parquet", columns=EXTRA_SRC)
        snap = snap.drop_duplicates("bridge_id")
        parts.append(snap[snap["bridge_id"].isin(grp["bridge_id"])])
    extras = pd.concat(parts, ignore_index=True).rename(columns={
        "STATE_CODE_001": "state", "DECK_COND_058": "deck_rating",
        "SUPERSTRUCTURE_COND_059": "super_rating", "SUBSTRUCTURE_COND_060": "sub_rating",
        "CULVERT_COND_062": "culvert_rating", "condition_rating": "min_rating"})
    out = df.merge(extras, on="bridge_id", how="left", validate="one_to_one")
    out["state"] = out["state"].astype("string")
    for c in RATING_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


train_data = pd.read_parquet(FINAL / "bridge_survival_train.parquet")
test_data = pd.read_parquet(FINAL / "bridge_survival_test_aligned.parquet")

# Add the condition ratings recorded in each bridge's first Good year.
train_data = add_baseline_extras(train_data)
test_data = add_baseline_extras(test_data)

BASE_NUM = ["bridge_age", "ADT_029", "STRUCTURE_LEN_MT_049"]
BASE_CAT = ["STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B"]

print("Training records:", f"{len(train_data):,}", "| events:", f"{int(train_data['event'].sum()):,}")
print("Testing records:", f"{len(test_data):,}", "| events:", f"{int(test_data['event'].sum()):,}")
print("\nOverall rating in first Good year (train):")
print(train_data["min_rating"].value_counts(dropna=False).sort_index().to_string())
print("Missing ratings (train / test):", int(train_data["min_rating"].isna().sum()), "/", int(test_data["min_rating"].isna().sum()))

Training records: 498,875 | events: 237,197
Testing records: 171,766 | events: 22,396

Overall rating in first Good year (train):
min_rating
7.0    248877
8.0    174437
9.0     75561
Missing ratings (train / test): 0 / 0


## 2. Prepare model inputs

In [3]:
def make_design(tr, other, num, cat=(), rating_cols=(), log_cols=(), age_sq=False, scale=False):
    # Build model inputs from `tr`, applying exactly the same fitted choices to `other`.
    X_tr = pd.DataFrame(index=tr.index)
    X_ot = pd.DataFrame(index=other.index)
    cont = []
    for c in list(num) + list(rating_cols):
        t = pd.to_numeric(tr[c], errors="coerce")
        o = pd.to_numeric(other[c], errors="coerce")
        if c in rating_cols:                         # remember where a rating was not recorded
            X_tr[c + "_na"] = t.isna().astype(int)
            X_ot[c + "_na"] = o.isna().astype(int)
        med = t.median()
        med = 0.0 if not np.isfinite(med) else med
        t, o = t.fillna(med), o.fillna(med)
        if age_sq and c == "bridge_age":
            X_tr["bridge_age_sq"], X_ot["bridge_age_sq"] = t ** 2, o ** 2
            cont.append("bridge_age_sq")
        if c in log_cols:
            t, o = np.log1p(t.clip(lower=0)), np.log1p(o.clip(lower=0))
        X_tr[c], X_ot[c] = t, o
        cont.append(c)
    if scale:                                        # standardise using training values only
        for c in cont:
            mu, sd = X_tr[c].mean(), X_tr[c].std()
            if sd and np.isfinite(sd):
                X_tr[c] = (X_tr[c] - mu) / sd
                X_ot[c] = (X_ot[c] - mu) / sd
    for c in cat:
        t = tr[c].astype("string").fillna("__missing__")
        o = other[c].astype("string").fillna("__missing__")
        counts = t.value_counts()
        keep = set(counts[counts >= RARE_MIN].index)
        t = t.where(t.isin(keep), "__other__")
        o = o.where(o.isin(keep), "__other__")
        cats = sorted(set(t.astype(str)) | {"__other__", "__missing__"})
        for series, X in ((t, X_tr), (o, X_ot)):
            d = pd.get_dummies(
                pd.Series(pd.Categorical(series.astype(str), categories=cats), index=series.index),
                prefix=c, drop_first=True, dtype=int)
            for col in d.columns:
                X[col] = d[col]
    X_ot = X_ot[X_tr.columns]
    keep_cols = [c for c in X_tr.columns if X_tr[c].nunique() > 1]   # drop constant columns
    return X_tr[keep_cols].astype(float), X_ot[keep_cols].astype(float)

X_train_df, X_test_df = make_design(
    train_data, test_data, num=BASE_NUM, cat=BASE_CAT, rating_cols=RATING_COLS, scale=False)

X_train = X_train_df.to_numpy(np.float32)
X_test = X_test_df.to_numpy(np.float32)
y_train = Surv.from_arrays(event=train_data["event"].to_numpy(bool), time=train_data["duration"].to_numpy(float))

print("Model inputs:", X_train.shape[1])
print("Training:", X_train.shape, "| Testing:", X_test.shape, "| training events:", int(y_train["event"].sum()))
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()
assert len(X_test) == len(test_data)

Model inputs: 40
Training: (498875, 40) | Testing: (171766, 40) | training events: 237197


## 3. Train the forest

In [4]:
t0 = time.time()
rsf_model = RandomSurvivalForest(
    n_estimators=N_ESTIMATORS, min_samples_split=MIN_SAMPLES_SPLIT, min_samples_leaf=MIN_SAMPLES_LEAF,
    max_depth=MAX_DEPTH, max_features="sqrt", max_samples=0.5, n_jobs=N_JOBS,
    random_state=RANDOM_STATE, low_memory=False)
rsf_model.fit(X_train, y_train)
print("Random Survival Forest trained in", round((time.time() - t0) / 60, 1), "minutes;",
      "trees:", len(rsf_model.estimators_), "| training records:", f"{len(X_train):,}")

Random Survival Forest trained in 7.3 minutes; trees: 50 | training records: 498,875


## 4. Median remaining years for **all** test bridges

For each bridge the forest gives a survival curve. The median remaining time is the first time after `entry_duration` at which the curve has fallen to half of its value at entry. If it never does within the observed training time, the median is left blank. This step is slow (it builds one curve per bridge), so progress is printed.

In [5]:
n_test = len(X_test)
entry = pd.to_numeric(test_data["entry_duration"], errors="coerce").to_numpy(dtype=float)
max_time = float(rsf_model.unique_times_[-1])
median_total = np.full(n_test, np.nan)

for start in range(0, n_test, BATCH_SIZE):
    stop = min(start + BATCH_SIZE, n_test)
    functions = rsf_model.predict_survival_function(X_test[start:stop])
    for local_i, fn in enumerate(functions):
        i = start + local_i
        e = entry[i]
        if not np.isfinite(e) or e < 0 or e > max_time:
            continue
        s_entry = float(fn(e))
        if not np.isfinite(s_entry) or s_entry <= 0:
            continue
        grid = fn.x
        pos = np.searchsorted(grid, e, side="left")
        if pos >= len(grid):
            continue
        crossings = np.flatnonzero(fn.y[pos:] <= 0.5 * s_entry)
        if crossings.size:
            median_total[i] = float(grid[pos + crossings[0]])
    if stop % 10000 < BATCH_SIZE or stop == n_test:
        print(f"Median predictions processed: {stop:,} / {n_test:,}")

remaining = np.where(np.isfinite(median_total), np.maximum(median_total - entry, 0), np.nan)
print("Median estimable:", f"{int(np.isfinite(remaining).sum()):,}", "/", f"{n_test:,}")
print(pd.Series(remaining).describe().round(2).to_string())

Median predictions processed: 10,000 / 171,766
Median predictions processed: 20,000 / 171,766
Median predictions processed: 30,000 / 171,766
Median predictions processed: 40,000 / 171,766
Median predictions processed: 50,000 / 171,766
Median predictions processed: 60,000 / 171,766
Median predictions processed: 70,000 / 171,766
Median predictions processed: 80,000 / 171,766
Median predictions processed: 90,000 / 171,766
Median predictions processed: 100,000 / 171,766
Median predictions processed: 110,000 / 171,766
Median predictions processed: 120,000 / 171,766
Median predictions processed: 130,000 / 171,766
Median predictions processed: 140,000 / 171,766
Median predictions processed: 150,000 / 171,766
Median predictions processed: 160,000 / 171,766
Median predictions processed: 170,000 / 171,766
Median predictions processed: 171,766 / 171,766
Median estimable: 49,647 / 171,766
count    49647.00
mean        16.71
std          4.91
min          5.00
25%         13.00
50%         17.00
75

## 5. Risk scores and full-test C-index

In [6]:
risk_scores = np.full(n_test, np.nan)
for start in range(0, n_test, 20000):
    stop = min(start + 20000, n_test)
    risk_scores[start:stop] = rsf_model.predict(X_test[start:stop])
assert np.isfinite(risk_scores).all(), "Non-finite risk scores; investigate before evaluation"

test_times = pd.to_numeric(test_data["test_duration"], errors="coerce").to_numpy(dtype=float)
test_events = test_data["event"].astype(bool).to_numpy()
assert np.isfinite(test_times).all() and (test_times >= 0).all()

full_test_cindex = float(concordance_index(test_times, -risk_scores, event_observed=test_events))
print("RSF FULL-TEST hazard-ranking C-index:", round(full_test_cindex, 4))
print("Bridges evaluated:", f"{n_test:,}", "| observed events:", f"{int(test_events.sum()):,}")
print("This is a ranking metric, not an error in years, and does not adjust for delayed entry.")

predictions = test_data[["bridge_id", "entry_duration", "test_duration", "event"]].copy().reset_index(drop=True)
predictions["median_total_years_since_first_good"] = median_total
predictions["median_remaining_years"] = remaining
predictions["median_estimable"] = np.isfinite(remaining)
predictions["median_within_training_support"] = np.isfinite(remaining) & (median_total <= max_time)
predictions["risk_score"] = risk_scores         # higher = earlier predicted deterioration (same convention in all models)
predictions.to_csv(RESULTS / "time_rsf_improved.csv", index=False)

metrics = pd.DataFrame([{
    "model": "Random Survival Forest (improved)",
    "inputs": int(X_train.shape[1]),
    "train_records": len(train_data), "test_records": n_test, "test_events": int(test_events.sum()),
    "full_test_c_index_risk_score": full_test_cindex,
    "median_estimable": int(np.isfinite(remaining).sum()),
    "median_unavailable": int(np.isnan(remaining).sum()),
    "max_training_time_support": max_time,
}])
metrics.to_csv(RESULTS / "rsf_improved_metrics.csv", index=False)
print("Saved:", RESULTS / "time_rsf_improved.csv", "and", RESULTS / "rsf_improved_metrics.csv")

RSF FULL-TEST hazard-ranking C-index: 0.6179
Bridges evaluated: 171,766 | observed events: 22,396
This is a ranking metric, not an error in years, and does not adjust for delayed entry.
Saved: /Users/annie/Downloads/bridge-deterioration/results/time_rsf_improved.csv and /Users/annie/Downloads/bridge-deterioration/results/rsf_improved_metrics.csv
